# pandas 入门：表格数据与金融时间序列

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Python-Pandas/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

## 一、安装与导入

In [ ]:
import numpy as np
import pandas as pd    #导入pandas模块，习惯上命名为pd

print(pd.__version__)    #查看pandas版本信息

### Series

In [ ]:
price = pd.Series([10.5, 10.8, 10.6, 11.2], index=["周一", "周二", "周三", "周四"], name="收盘价")
print(price)
print("周三的收盘价：", price["周三"])

### DataFrame

最常用的方法是用字典创建，字典的键就是列名：

In [ ]:
stocks = pd.DataFrame({
    "代码": ["600519", "000001", "601318", "000858"],
    "名称": ["贵州茅台", "平安银行", "中国平安", "五粮液"],
    "行业": ["白酒", "银行", "保险", "白酒"],
    "市盈率": [28.5, 4.6, 8.9, 18.2],
})
print(stocks)

### 带日期索引的 DataFrame

金融数据最常见的形式是“日期 × 资产”的价格表。为了让代码在任何电脑上都能运行，这里用 NumPy 的随机数模拟三只股票 2023 年的日收盘价（随机数的用法见 [NumPy 入门](https://wxrkz123.github.io/blog/Python-Numpy/)第十一节）：

In [ ]:
rng = np.random.default_rng(2119)    #固定随机种子，保证每次结果相同
dates = pd.bdate_range("2023-01-02", "2023-12-29")    #2023年的工作日
mu = [0.0006, 0.0002, 0.0009]    #三只股票的日均收益率
sigma = [0.018, 0.011, 0.026]    #三只股票的日波动率
daily_ret = rng.normal(mu, sigma, size=(len(dates), 3))

prices = pd.DataFrame(100 * np.exp(daily_ret.cumsum(axis=0)),    #从100元开始累积
                      index=dates, columns=["股票A", "股票B", "股票C"]).round(2)
prices.index.name = "日期"
print(prices.head())

## 三、读取与保存数据

实际工作中，数据一般来自文件。pandas 可以读写几乎所有常见的格式：

（示意代码，不在笔记本中运行）

```python
pd.read_csv("文件名.csv")    #读取CSV文件
pd.read_excel("文件名.xlsx")    #读取Excel文件（需要安装openpyxl）
df.to_csv("文件名.csv")    #保存为CSV文件
df.to_excel("文件名.xlsx")    #保存为Excel文件
```

把上面的价格表存成 CSV，再读回来：

In [ ]:
prices.to_csv("prices.csv", encoding="utf-8-sig")    #utf-8-sig让Excel打开时中文不乱码
df = pd.read_csv("prices.csv", index_col="日期", parse_dates=True)
print(df.index[:3])

## 四、查看数据

拿到一张表，先看看它长什么样：

（示意代码，不在笔记本中运行）

```python
df.head(n)    #前n行，默认5行
df.tail(n)    #后n行
df.shape    #(行数, 列数)
df.columns    #列名
df.index    #行索引
df.dtypes    #每一列的数据类型
df.info()    #总体信息：行数、列名、类型、非缺失值个数
df.describe()    #描述统计：均值、标准差、最小值、分位数、最大值
```

In [ ]:
print(df.shape)
print(df.describe().round(2))

## 五、索引、切片与筛选

pandas 取数据的方法比较多，一开始容易混。常用的是这几种：

（示意代码，不在笔记本中运行）

```python
df["列名"]    #取一列，得到Series
df[["列1", "列2"]]    #取多列，得到DataFrame
df.loc[行标签, 列标签]    #按标签取
df.iloc[行位置, 列位置]    #按位置取（从0开始）
df[条件]    #按条件筛选行
```

`loc` 用的是标签（名字、日期），`iloc` 用的是位置（第几行、第几列），这是最容易搞混的地方。

In [ ]:
print(df.loc["2023-06-01":"2023-06-05", ["股票A", "股票C"]])    #日期切片，两端都包含
print(df.iloc[-1])    #最后一行，即年末的价格

按条件筛选，比如找出股票 C 价格超过 155 元的日子：

In [ ]:
print(df[df["股票C"] > 155])
print("\n白酒股：")
print(stocks[stocks["行业"] == "白酒"])

### 新增列

列与列之间的计算和 NumPy 数组一样，是逐元素进行的：

In [ ]:
stocks["每股收益"] = (50 / stocks["市盈率"]).round(2)    #假设股价都是50元
stocks["估值"] = np.where(stocks["市盈率"] > 15, "偏高", "偏低")
print(stocks)

### 排序

（示意代码，不在笔记本中运行）

```python
df.sort_values("列名", ascending=False)    #按某一列降序排序
df.sort_index()    #按索引排序
```

In [ ]:
print(stocks.sort_values("市盈率")[["名称", "市盈率"]])

### 缺失值

真实数据中经常有缺失值，比如停牌的日子没有价格。pandas 用 `NaN` 表示缺失值：

（示意代码，不在笔记本中运行）

```python
df.isna()    #判断每个值是否缺失
df.isna().sum()    #每一列缺失值的个数
df.dropna()    #删除有缺失值的行
df.fillna(值)    #用指定的值填充
df.ffill()    #用前一个值填充（前向填充）
```

In [ ]:
halted = df["股票A"].copy()
halted["2023-03-06":"2023-03-08"] = np.nan    #假设股票A停牌了三天
print("缺失值个数：", halted.isna().sum())
print(halted.ffill()["2023-03-03":"2023-03-09"])

### 分组汇总

`groupby` 是 pandas 最强大的功能之一：按某一列分组，然后对每一组计算统计量。

In [ ]:
print(stocks.groupby("行业")["市盈率"].agg(["count", "mean", "max"]))

### 合并表格

两张表有共同的列时，可以用 `merge` 合并，类似 Excel 里的 VLOOKUP：

In [ ]:
holdings = pd.DataFrame({"代码": ["600519", "601318", "000651"], "持仓股数": [100, 500, 300]})
merged = holdings.merge(stocks[["代码", "名称", "行业"]], on="代码", how="left")
print(merged)

### 收益率：shift 与 pct_change

In [ ]:
ret = df.pct_change()    #日收益率 =（今天价格 / 昨天价格）- 1
ret2 = df / df.shift(1) - 1    #和上面完全一样，shift(1)把数据整体往后挪一天
print(ret.head(3).round(4))
print("两种方法相同：", np.allclose(ret.dropna(), ret2.dropna()))

### 重采样：从日数据到月数据

In [ ]:
monthly_price = df.resample("ME").last()    #每个月最后一个交易日的价格
monthly_ret = monthly_price.pct_change()
print(monthly_ret.head(4).round(4))

### 滚动窗口：移动平均

In [ ]:
ma = pd.DataFrame({
    "收盘价": df["股票A"],
    "MA20": df["股票A"].rolling(20).mean(),    #20日均线
    "MA60": df["股票A"].rolling(60).mean(),    #60日均线
})
print(ma.tail(3).round(2))

## 九、收益与风险：年化收益率、波动率与夏普比率

In [ ]:
ret = df.pct_change().dropna()
summary = pd.DataFrame({
    "全年收益率": df.iloc[-1] / df.iloc[0] - 1,
    "年化波动率": ret.std() * np.sqrt(252),    #一年约252个交易日
})
summary["夏普比率"] = (ret.mean() * 252 - 0.02) / summary["年化波动率"]    #假设无风险利率2%
print(summary.round(3))

## 十、最大回撤

最大回撤是指从某个最高点跌到之后最低点的最大跌幅，也就是“最倒霉的时候买入，最多会亏多少”：

In [ ]:
running_max = df.cummax()    #截至每一天的历史最高价
drawdown = df / running_max - 1    #当天相对于历史最高价跌了多少
print("最大回撤：")
print(drawdown.min().round(3))
print("\n股票C最大回撤发生在：", drawdown["股票C"].idxmin().date())

## 十一、相关系数

In [ ]:
print(ret.corr().round(3))